# Tutorial 05: diabatic states – learning the diabatic Hamiltonian, dynamics in SHARC

*Author: Rhyan Barrett*

A diabatic FieldMACE model learns the elements of the diabatic Hamiltonian H(R, MM charges) and their
gradients. Adiabatic energies, gradients and NACs (also on the MM atoms) follow from diagonalising H. The SHARC
interface does this every step, so SHARC runs unchanged, with NACs or with wave-function overlaps.
CH₂S in water again, with diabatic labels from SHARC's LVC model. Details: [README](README.md).

In [ ]:
import os, subprocess, sys
import numpy as np
from IPython.display import Image, display

QUICK = os.environ.get("FIELDMACE_TUTORIAL_QUICK") == "1"   # tiny settings for automatic checks
os.environ.setdefault("OMP_NUM_THREADS", "4")

def run(cmd, env=None):
    """Run a tutorial step and show its output (stops the notebook if the step fails)."""
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                       env={**os.environ, **(env or {})})
    lines = [l for l in p.stdout.splitlines() if "Warning" not in l and "warnings.warn" not in l]
    print("\n".join(lines[-60:]))
    if p.returncode:
        raise RuntimeError(f"step failed: {cmd}")
assert os.environ.get("SHARC"), "set $SHARC (SHARC 4 bin directory) before starting Jupyter"

## Step 1: sampling (as tutorial 02)

In [ ]:
run(f"bash 01_sample_configurations.sh --n-frames {80 if QUICK else 1000} --equil-ps {0.5 if QUICK else 20} --n-initconds {4 if QUICK else 10}")

## Step 2: diabatic labels

`REF_energy` holds the 10 elements `[H_00 H_11 H_22 H_33 | H_01 H_02 H_03 H_12 H_13 H_23]`; `REF_forces` and
`REF_mm_forces` hold −dH/dR of every element on the atoms and on the point charges.

In [ ]:
run("python 02_label_diabatic.py")

In [ ]:
import ase.io
import matplotlib.pyplot as plt
from mace.tools.diabatic import unpack
frames = ase.io.read("data/train.xyz", index=":")
x = [f for f in frames if f.info.get("config_type") == "solvated"][0]
for k in ("REF_energy", "REF_forces", "REF_mm_forces", "REF_dipoles", "REF_socs"):
    print(f"{k:14s}", np.shape(x.info[k]))
H = unpack(np.ravel(x.info["REF_energy"]), 4)
print("diabatic Hamiltonian (eV, diagonal relative to H_00):"); print(np.round(H - H[0, 0] * np.eye(4), 4))
sol = np.array([np.ravel(f.info["REF_energy"]) for f in frames if f.info.get("config_type") == "solvated"])
plt.figure(figsize=(5, 3))
plt.hist(1000 * sol[:, 4], bins=40, alpha=0.6, label="H_01 (singlets)")
plt.hist(1000 * sol[:, 9], bins=40, alpha=0.6, label="H_23 (triplets)")
plt.xlabel("diabatic coupling (meV)"); plt.legend(); plt.show()

## Step 3: training (`--diabatic=True --n_energies=4`)

In [ ]:
env = {"EPOCHS": "2", "CHANNELS": "8"} if QUICK else {}
run("bash 03_train.sh", env=env)

## Step 4: evaluation – diabatic elements, and the adiabatic energies and NACs that follow

In [ ]:
run("python 04_evaluate.py")
display(Image("results/parity_diabatic.png"))

In [ ]:
from mace.calculators import MACECalculator
calc = MACECalculator("models/ch2s_diabatic.model", device="cpu", diabatic_blocks=[[0, 1], [2, 3]])
t = [f for f in ase.io.read("data/test.xyz", index=":") if "mm_charges" in f.info][0]
a = t.copy(); a.info = {"mm_positions": t.info["mm_positions"], "mm_charges": t.info["mm_charges"]}
calc.calculate(a)
r = calc.results
print("adiabatic energies (eV):", r["energies"], " LVC:", np.ravel(t.info["REF_adiabatic_energy"]))
print("NAC S0-S1 on the QM atoms (1/A):\n", r["nacs"][:, 0])
print("largest NAC S0-S1 on a water charge (1/A):", np.abs(r["mm_nacs"][:, 0]).max())

## Step 5: surface hopping in SHARC, FieldMACE (diabatic) vs LVC

Gas phase and QM/MM (SHARC_QMMM + OpenMM), trajectories starting in T2. First with NACs (`coupling ddr`),
then with wave-function overlaps (`coupling overlap`), which only diabatic models can give.

In [ ]:
env = {"NTRAJ": "1", "TMAX": "2"} if QUICK else {}
run("bash 05_sharc_dynamics.sh", env=env)
display(Image("results/ddr_gas/populations.png"))
display(Image("results/ddr_qmmm/populations.png"))

In [ ]:
run("bash 05_sharc_dynamics.sh", env={**env, "MODES": "gas", "COUPLING": "overlap"})
display(Image("results/overlap_gas/populations.png"))